<a href="https://colab.research.google.com/github/barbarajunq131/Programas/blob/main/Indice_de_sustentabilidade_local_e_justica_racial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install basedosdados

1. Importar bibliotecas


In [22]:
import basedosdados as bd
import pandas as pd
import numpy as np

2. Configurações

In [23]:
project_id = "indiceequidade" # projeto no GCP

municipios_ids = "('1501402', '1506807', '2927408', '2910800', '5208707', '5201405', '3550308', '3509502', '4106902', '4113700')"

mapa_cidades = {
    '1501402': 'Belém (PA)', '1506807': 'Santarém (PA)',
    '2927408': 'Salvador (BA)', '2910800': 'Feira de Santana (BA)',
    '5208707': 'Goiânia (GO)', '5201405': 'Aparecida de Goiânia (GO)',
    '3550308': 'São Paulo (SP)', '3509502': 'Campinas (SP)',
    '4106902': 'Curitiba (PR)', '4113700': 'Londrina (PR)'
}

print("Iniciando o Pipeline de Dados...")
print("1/4: Extraindo microdados brutos do Censo 2010 (Isso leva uns 2 a 3 minutos)...")

Iniciando o Pipeline de Dados...
1/4: Extraindo microdados brutos do Censo 2010 (Isso leva uns 2 a 3 minutos)...


3. Extração das variáveis

In [24]:
query_master = f"""
WITH base_pessoas AS (
    SELECT
        id_municipio,
        controle,
        -- Raça: Forçando para STRING para evitar erro de INT
        CASE
            WHEN CAST(v0606 AS STRING) = '1' THEN 'Branca'
            WHEN CAST(v0606 AS STRING) IN ('2', '4') THEN 'Negra'
        END as raca,

        -- Renda: Transformando em Float e anulando se for 0.0 (Numeric)
        NULLIF(CAST(v6526 AS FLOAT64), 0.0) as renda,

        -- Escolaridade: Filtrando até 15 anos para ignorar erros do IBGE
        CASE WHEN CAST(v0633 AS FLOAT64) < 16.0 THEN CAST(v0633 AS FLOAT64) END as escolaridade,

        -- Analfabetismo: 1=Sabe ler, 2=Não sabe
        CASE WHEN CAST(v0627 AS STRING) = '2' THEN 1 WHEN CAST(v0627 AS STRING) = '1' THEN 0 END as analfabeto,

        -- Desemprego: 1=Ocupada, 2=Desocupada
        CASE WHEN CAST(v0635 AS STRING) = '2' THEN 1 WHEN CAST(v0635 AS STRING) = '1' THEN 0 END as desempregado

    FROM `basedosdados.br_ibge_censo_demografico.microdados_pessoa_2010`
    WHERE id_municipio IN {municipios_ids} AND CAST(v0606 AS STRING) IN ('1', '2', '4')
),
base_domicilio AS (
    SELECT
        id_municipio,
        controle,
        -- Esgoto e Lixo: 1 e 2 são as respostas adequadas/coletadas
        CASE WHEN CAST(v0207 AS STRING) IN ('1', '2') THEN 1 ELSE 0 END as esgoto_adequado,
        CASE WHEN CAST(v0209 AS STRING) IN ('1', '2') THEN 1 ELSE 0 END as lixo_adequado
    FROM `basedosdados.br_ibge_censo_demografico.microdados_domicilio_2010`
    WHERE id_municipio IN {municipios_ids}
)
SELECT
    p.id_municipio,
    p.raca,
    AVG(p.renda) as media_renda,
    AVG(p.escolaridade) as media_escolaridade,
    AVG(p.analfabeto) as taxa_analfabetismo,
    AVG(p.desempregado) as taxa_desemprego,
    AVG(d.esgoto_adequado) as taxa_esgoto,
    AVG(d.lixo_adequado) as taxa_lixo
FROM base_pessoas p
JOIN base_domicilio d ON p.id_municipio = d.id_municipio AND p.controle = d.controle
GROUP BY 1, 2
"""

df_bruto = bd.read_sql(query=query_master, billing_project_id=project_id)

print("2/4: Dados extraídos! Pivotando e calculando os gaps raciais...")

Downloading: 100%|██████████|
2/4: Dados extraídos! Pivotando e calculando os gaps raciais...


3. TRATAMENTO DE DADOS E CÁLCULO DAS RAZÕES (GAPS)

In [25]:
#separando brancos e negros
df_pivot = df_bruto.pivot(index='id_municipio', columns='raca')
df_pivot.columns = [f"{col[0]}_{col[1][0]}" for col in df_pivot.columns]
df_pivot['Municipio'] = df_pivot.index.map(mapa_cidades)
df_pivot.set_index('Municipio', inplace=True)

# Gaps Raciais
df_gaps = pd.DataFrame(index=df_pivot.index)
df_gaps['R_Renda'] = df_pivot['media_renda_N'] / df_pivot['media_renda_B']
df_gaps['R_Escol'] = df_pivot['media_escolaridade_N'] / df_pivot['media_escolaridade_B']
df_gaps['R_Esgoto'] = df_pivot['taxa_esgoto_N'] / df_pivot['taxa_esgoto_B']
df_gaps['R_Lixo'] = df_pivot['taxa_lixo_N'] / df_pivot['taxa_lixo_B']
df_gaps['R_Desemp'] = df_pivot['taxa_desemprego_N'] / df_pivot['taxa_desemprego_B']
df_gaps['R_Analf'] = df_pivot['taxa_analfabetismo_N'] / df_pivot['taxa_analfabetismo_B']

print("3/4: Aplicando a normalização e calculando Entropia de Shannon...")

3/4: Aplicando a normalização e calculando Entropia de Shannon...


4. NORMALIZAÇÃO E CÁLCULO DO ISL (TN e TE)

In [26]:
df_norm = pd.DataFrame(index=df_gaps.index)

for var in ['R_Renda', 'R_Escol', 'R_Esgoto', 'R_Lixo']:
    df_norm[f'Z_{var}'] = ((df_gaps[var] - df_gaps[var].min()) / (df_gaps[var].max() - df_gaps[var].min())) * 100

for var in ['R_Desemp', 'R_Analf']:
    df_norm[f'Z_{var}'] = ((df_gaps[var].max() - df_gaps[var]) / (df_gaps[var].max() - df_gaps[var].min())) * 100

k = 6
lambda_peso = 0.3

df_isl = pd.DataFrame(index=df_norm.index)
df_isl['Termo_Nivel_TN'] = df_norm.mean(axis=1)

def calcular_te(linha_z):
    linha_z = np.where(linha_z == 0, 1e-10, linha_z)
    z_prop = linha_z / np.sum(linha_z)
    return -(100 / np.log(k)) * np.sum(z_prop * np.log(z_prop))

df_isl['Termo_Eq_TE'] = df_norm.apply(calcular_te, axis=1)
df_isl['ISL_FINAL'] = ((1 - lambda_peso) * df_isl['Termo_Nivel_TN']) + (lambda_peso * df_isl['Termo_Eq_TE'])

print("4/4: Salvando resultados...")

4/4: Salvando resultados...


5. EXIBIÇÃO E EXPORTAÇÃO

In [28]:
df_completo = pd.concat([df_pivot, df_gaps, df_isl], axis=1)
df_completo.sort_values(by='ISL_FINAL', ascending=False, inplace=True)

print("\n" + "="*60)
print("   RANKING FINAL DO ISL COM DADOS REAIS DO IBGE")
print("="*60)
display(df_completo[['Termo_Nivel_TN', 'Termo_Eq_TE', 'ISL_FINAL']].round(2))

df_completo.to_excel('TCC_ISL_Completo.xlsx')
print("\n✅ SUCESSO! O arquivo 'TCC_ISL_Completo.xlsx' foi gerado.")


   RANKING FINAL DO ISL COM DADOS REAIS DO IBGE


,Termo_Nivel_TN,Termo_Eq_TE,ISL_FINAL
Municipio,,,
Aparecida de Goiânia (GO),72.44,90.80,77.95
Belém (PA),63.73,97.83,73.96
Londrina (PR),53.08,88.32,63.65
Goiânia (GO),46.48,85.85,58.29
Curitiba (PR),47.48,74.65,55.63
Salvador (BA),44.91,77.77,54.77
São Paulo (SP),44.93,76.20,54.31
Santarém (PA),43.63,74.15,52.78
Feira de Santana (BA),36.67,86.33,51.56



✅ SUCESSO! O arquivo 'TCC_ISL_Completo.xlsx' foi gerado.
